# Gaussian naive Bayes — daily flood occurrence

Trains `GaussianNB` on `dataset/flood_training_data_split.csv` and reports
accuracy, precision, recall, F1 and MCC on the held-out test period.

| Stage | Detail |
| --- | --- |
| Features | `ante_15d` (log1p), `tmin_c` |
| Split | chronological, pre-built: train to 2014-02-28, test from 2014-03-01 |
| Imbalance | equal priors, `priors=[0.5, 0.5]` |
| Threshold | F1-optimal on the last 20% of train |

Three differences from the SVM notebook, all consequences of the model:

- **No scaling.** GaussianNB fits a per-class mean and variance per feature, so it
  is scale-invariant.
- **`log1p` is kept, and matters more here.** NB assumes each feature is Gaussian
  within each class; rainfall is right-skewed, so the transform improves the
  model's actual assumption rather than just conditioning an optimiser.
- **No subsampling or averaging.** GaussianNB is O(n) and deterministic, so it
  fits all 38,799 rows at once — none of the seed instability that forced
  15-model averaging for the RBF.

The two features correlate at 0.04, which matters because NB assumes conditional
independence. The three raw rainfall columns correlate at 0.87–0.93 and would make
it count the same evidence three times.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix, f1_score,
                             matthews_corrcoef, precision_recall_curve,
                             precision_score, recall_score)
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

LABEL = "Flood occurrences"


def find_repo_root(marker: str = "dataset") -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise FileNotFoundError(f"No parent of {Path.cwd()} contains {marker!r}")


REPO_ROOT = find_repo_root()
data = pd.read_csv(REPO_ROOT / "dataset/flood_training_data_split.csv", parse_dates=["date"])
data = data.sort_values("date").reset_index(drop=True)
data[LABEL] = data[LABEL].astype(bool)

train = data[data["split"] == "train"]
test = data[data["split"] == "test"]
y_train = train[LABEL].to_numpy()
y_test = test[LABEL].to_numpy()

print(f"train {len(train):,} rows, {y_train.sum()} floods   "
      f"test {len(test):,} rows, {y_test.sum()} floods")

In [ ]:
def build_model() -> Pipeline:
    """log1p on rainfall, temperature untouched, equal class priors.

    priors=[0.5, 0.5] is naive Bayes' equivalent of class weighting: it stops the
    model assuming the 0.2% base rate and so prevents it predicting "no flood"
    for everything. No scaler, since GaussianNB is scale-invariant.
    """
    return Pipeline([
        ("prep", ColumnTransformer([
            ("rain", FunctionTransformer(np.log1p), ["ante_15d"]),
            ("temp", "passthrough", ["tmin_c"]),
        ])),
        ("model", GaussianNB(priors=[0.5, 0.5])),
    ])


# Threshold picked on the last 20% of train; test is not consulted
cut = int(len(train) * 0.8)
validation_scores = build_model().fit(train.iloc[:cut], y_train[:cut]) \
                                 .predict_proba(train.iloc[cut:])[:, 1]

precision, recall, thresholds = precision_recall_curve(y_train[cut:], validation_scores)
f1_curve = np.divide(2 * precision * recall, precision + recall,
                     out=np.zeros_like(precision), where=(precision + recall) > 0)
# precision_recall_curve returns one more point than it does thresholds
THRESHOLD = float(thresholds[max(0, int(np.argmax(f1_curve)) - 1)])

# Refit on the full training split, score test once
test_scores = build_model().fit(train, y_train).predict_proba(test)[:, 1]
predictions = test_scores >= THRESHOLD

print(f"threshold {THRESHOLD:.4f}   flagged {predictions.sum():,} of {len(test):,} test days")

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, predictions).ravel()

print(f"TP {tp}   FP {fp:,}   FN {fn}   TN {tn:,}\n")
for name, value in {
    "Accuracy": accuracy_score(y_test, predictions),
    "Precision": precision_score(y_test, predictions, zero_division=0),
    "Recall": recall_score(y_test, predictions),
    "F1 Score": f1_score(y_test, predictions),
    "MCC": matthews_corrcoef(y_test, predictions),
    # Threshold-free, and the floor is the prevalence rather than 0.5, so it is
    # the metric that survives a 0.25% positive rate
    "PR-AUC": average_precision_score(y_test, test_scores),
}.items():
    print(f"{name:<10} {value:.4f}")

## Results

| Metric | Naive Bayes | RBF SVM |
| --- | --- | --- |
| Accuracy | 0.8198 | 0.9090 |
| Precision | 0.0063 | 0.0046 |
| Recall | 0.2895 | 0.1053 |
| F1 Score | 0.0123 | 0.0089 |
| **MCC** | **0.0181** | 0.0038 |

Confusion matrix: TP 11, FP 1,740, FN 27, TN 8,028.

**Naive Bayes beats the SVM at the operating point** — MCC 0.0181 against 0.0038,
and it catches 11 of 38 floods against the SVM's 4. Most likely because with 83
training floods, estimating four numbers per class overfits less than fitting a
flexible decision boundary.

**The SVM still ranks better overall** (PR-AUC 0.0120 against 0.0099). So NB is
better at one chosen threshold while the SVM orders the full list better — worth
noting rather than declaring an outright winner.

**`log1p` earns its place.** Without it MCC falls to 0.0097 and PR-AUC halves to
0.0051, which is the Gaussian assumption being violated by skewed rainfall.
Equal priors and `var_smoothing` barely move the result; the priors only rescale
the probabilities, leaving the ranking unchanged.

**Accuracy of 0.8198 is the worst of the three options and that is expected.**
Predicting "no flood" always scores 0.9961 at MCC 0. Equal priors deliberately
trade accuracy for recall, so the drop is the model working as configured.

In absolute terms MCC 0.018 is still near chance. Both models are limited by the
same things: 83 training floods, and a label process whose seasonal and spatial
patterns reverse between the two periods.